# Artificial Neural Network

### Importing Liberaries

In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler 
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

## Data preprocessing

#### Importing Dataset

In [3]:
df = pd.read_csv('../Data/P1-ANN/Churn_Modelling.csv')
X = df.iloc[:, 3:-1].values
y = df.iloc[:, -1].values

### Encoding Categorical data

#### Label Encoding the "Gender" column

In [4]:
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

#### One-Hot Encoding Of Geography

In [5]:
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(sparse_output= False), [1])], remainder= 'passthrough')
X = ct.fit_transform(X)

In [6]:
print(X)

[[1.0 0.0 0.0 ... 1 1 101348.88]
 [0.0 0.0 1.0 ... 0 1 112542.58]
 [1.0 0.0 0.0 ... 1 0 113931.57]
 ...
 [1.0 0.0 0.0 ... 0 1 42085.58]
 [0.0 1.0 0.0 ... 1 0 92888.52]
 [1.0 0.0 0.0 ... 1 0 38190.78]]


#### Splitting the Dataset into the training and testing set

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.25, random_state= 42, stratify= y)

#### Feature Scaling

In [8]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Building the ANN

#### Initializing the ANN

In [9]:
ann = tf.keras.models.Sequential()

# Adding the input layer and the first hidden layer
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

# Adding the second hidden layer
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

# Adding output layer
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# For non binary we will use softmax activation function 

## Training the ANN

In [10]:
ann.compile(optimizer = "adam", loss = 'binary_crossentropy', metrics = ['accuracy'])
# this is for binary and for non binary we will categorical_crossentropy 

ann.fit(X_train, y_train, batch_size= 32, epochs= 100)


Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 640us/step - accuracy: 0.6184 - loss: 0.6490
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 628us/step - accuracy: 0.7979 - loss: 0.4914
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 637us/step - accuracy: 0.8067 - loss: 0.4558
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 615us/step - accuracy: 0.8148 - loss: 0.4389
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 611us/step - accuracy: 0.8185 - loss: 0.4269
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 615us/step - accuracy: 0.8240 - loss: 0.4151
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 624us/step - accuracy: 0.8277 - loss: 0.4024
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 620us/step - accuracy: 0.8351 - loss: 0.3894
Epoch 9/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 628us/step - accuracy: 0.8388 - loss: 0.3787
Epoch 10/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 684us/step - accuracy: 0.8440 - loss: 0.3712
Epoch 11/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 620us/step - accuracy: 0.8464 - loss: 0.3656
Epoch 12/100
235/23

#### Predicting the Single Observation

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: $ 60000

Number of Products: 2

Does this customer have a credit card? Yes

Is this customer an Active Member: Yes

Estimated Salary: $ 50000

WIll the customer leave ??

In [11]:
print(ann.predict(sc.transform([[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])) > 0.5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
[[False]]


Therefore, our ANN model predicts that this customer stays in the bank!

Important note 1: Notice that the values of the features were all input in a double pair of square brackets. That's because the "predict" method always expects a 2D array as the format of its inputs. And putting our values into a double pair of square brackets makes the input exactly a 2D array.

Important note 2: Notice also that the "France" country was not input as a string in the last column but as "1, 0, 0" in the first three columns. That's because of course the predict method expects the one-hot-encoded values of the state, and as we see in the first row of the matrix of features X, "France" was encoded as "1, 0, 0". And be careful to include these values in the first three columns, because the dummy variables are always created in the first columns.

#### Predicting the Test set results 

In [12]:
y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)
print(np.concatenate((y_pred.reshape(len(y_pred), 1), y_test.reshape(len(y_test), 1)), 1))

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 500us/step
[[0 0]
 [0 0]
 [0 0]
 ...
 [1 1]
 [0 0]
 [0 0]]


#### Making confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[1909   82]
 [ 261  248]]


0.8628